# Governance - Antonio

# 0 Setup 




## Imports

In [ ]:
from pathlib import Path
import re
import pandas as pd



# 1. PII Minimization 

In [7]:
# Find repo root
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").is_dir() and ((p / ".git").exists() or (p / "README.md").exists() or (p / "src").is_dir()):
            return p
    for p in [start] + list(start.parents):
        if (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root. Open the notebook from inside the repo folder.")


REPO_ROOT = find_repo_root(Path.cwd())


#  Fixed input locations 
PII_PATH = REPO_ROOT / "data" / "quality" / "pii_inventory.csv"
ANALYSIS_PATH = REPO_ROOT / "data" / "curated" / "applications_analysis.csv"

#  Fixed output location 
OUT_DIR = REPO_ROOT / "reports" / "Governance" / "pii_minimization_study"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# Load data
pii = pd.read_csv(PII_PATH)
analysis = pd.read_csv(ANALYSIS_PATH)

print(f"Loaded pii_inventory rows = {len(pii)}")
print(f"Loaded analysis shape = {analysis.shape}")


# Presence matrix + direct PII list
# Expect columns: field_path, classification, present_in_raw, present_in_curated, present_in_analysis
presence_matrix = (
    pii.rename(columns={"field_path": "field_name", "classification": "pii_class"})
    .loc[:, ["field_name", "pii_class", "present_in_raw", "present_in_curated", "present_in_analysis"]]
    .sort_values(["pii_class", "field_name"])
)
presence_matrix.to_csv(OUT_DIR / "pii_presence_matrix.csv", index=False)

direct_pii_fields = (
    pii.loc[pii["classification"].astype(str).str.strip().str.lower() == "pii", "field_path"]
    .dropna()
    .astype(str)
    .str.strip()
    .sort_values()
    .unique()
    .tolist()
)
(Path(OUT_DIR / "direct_pii_fields_list.txt")).write_text("\n".join(direct_pii_fields), encoding="utf-8")


# Structural checks
direct_set = set(direct_pii_fields)
exact_present = [c for c in analysis.columns if c in direct_set]

direct_leaf = {f.split(".")[-1].lower() for f in direct_pii_fields}
leaf_present = [c for c in analysis.columns if c.lower() in direct_leaf]

print("Direct PII columns found (exact match):", exact_present)
print("Direct PII columns found (leaf-name match):", leaf_present)


# Full leakage scan on text columns only
scan_df = analysis.select_dtypes(include=["object"]).fillna("").astype(str)

patterns = {
    "email_like": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    "ipv4_like": re.compile(r"\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b"),
    "ssn_like_hyphen": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}

summary_rows = []
hit_details = []

if scan_df.shape[1] == 0:
    print("No text columns to scan.")
else:
    for name, pat in patterns.items():
        hits_bool = scan_df.apply(lambda col: col.str.contains(pat, regex=True, na=False))
        per_col_hits = hits_bool.sum(axis=0)

        cols_with_hits = per_col_hits[per_col_hits > 0].sort_values(ascending=False)
        cell_hits = int(per_col_hits.sum())

        summary_rows.append(
            {
                "pattern": name,
                "sample_size_rows": int(scan_df.shape[0]),
                "columns_scanned": int(scan_df.shape[1]),
                "cell_hits": cell_hits,
                "columns_with_hits": int((per_col_hits > 0).sum()),
            }
        )

        for col_name, n_hits in cols_with_hits.items():
            hit_details.append({"pattern": name, "column": col_name, "hits": int(n_hits)})

leak_summary = pd.DataFrame(
    summary_rows,
    columns=["pattern", "sample_size_rows", "columns_scanned", "cell_hits", "columns_with_hits"],
)
hit_details_df = pd.DataFrame(hit_details, columns=["pattern", "column", "hits"])

leak_summary.to_csv(OUT_DIR / "analysis_pii_leakage_scan_summary.csv", index=False)
hit_details_df.to_csv(OUT_DIR / "analysis_pii_leakage_by_column.csv", index=False)

print("\nLeakage scan summary:")
print(leak_summary if len(leak_summary) else "(no text columns scanned)")

if hit_details_df.shape[0] == 0:
    print("\nNo leakage hits by column (expected for a PII-safe analysis extract).")
else:
    print("\nLeakage hits by column (investigate these columns):")
    print(hit_details_df.sort_values(["pattern", "hits"], ascending=[True, False]).head(50))

Loaded pii_inventory rows = 10
Loaded analysis shape = (500, 16)
Direct PII columns found (exact match): []
Direct PII columns found (leaf-name match): []

Leakage scan summary:
           pattern  sample_size_rows  columns_scanned  cell_hits  \
0       email_like               500                6          0   
1        ipv4_like               500                6          0   
2  ssn_like_hyphen               500                6          0   

   columns_with_hits  
0                  0  
1                  0  
2                  0  

No leakage hits by column (expected for a PII-safe analysis extract).


# 2. Pseudonymization

In [6]:
try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root (no 'data/' folder found up the tree).")


REPO_ROOT = find_repo_root(Path.cwd())

ANALYSIS_PATH = REPO_ROOT / "data" / "curated" / "applications_analysis.csv"
CURATED_FULL_PATH = REPO_ROOT / "data" / "curated" / "applications_curated_full.csv"
OUT_DIR = REPO_ROOT / "reports" / "Governance" / "pseudonymization_study"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not ANALYSIS_PATH.exists():
    raise FileNotFoundError(f"Missing: {ANALYSIS_PATH}")
if not CURATED_FULL_PATH.exists():
    raise FileNotFoundError(f"Missing: {CURATED_FULL_PATH}")

analysis = pd.read_csv(ANALYSIS_PATH)

required_analysis = ["applicant_pseudo_id", "pseudo_id_source", "pseudo_id_fallback_used_flag"]
required_present = {c: (c in analysis.columns) for c in required_analysis}

direct_pii_cols = [
    "raw_applicant_full_name",
    "raw_applicant_email",
    "raw_applicant_ssn",
    "raw_applicant_ip_address",
    "raw_applicant_date_of_birth",
    "clean_email",
    "clean_date_of_birth",
]

direct_pii_present_in_analysis = [c for c in direct_pii_cols if c in analysis.columns]

# Normalize fallback flag to boolean when it comes as text
if "pseudo_id_fallback_used_flag" in analysis.columns and analysis["pseudo_id_fallback_used_flag"].dtype == object:
    analysis["pseudo_id_fallback_used_flag"] = (
        analysis["pseudo_id_fallback_used_flag"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

# Load curated_full (only needed cols)
curated_cols = list(pd.read_csv(CURATED_FULL_PATH, nrows=0).columns)
curated_cols_for_shot = [
    "application_id",
    "application_row_id",
    "is_canonical_for_analysis",
    "raw_applicant_full_name",
    "raw_applicant_email",
    "raw_applicant_ssn",
    "raw_applicant_ip_address",
    "raw_applicant_date_of_birth",
    "raw_applicant_zip_code",
    "clean_email",
    "clean_date_of_birth",
    "clean_zip_code",
    "clean_gender",
    "clean_annual_income",
]
usecols = [c for c in curated_cols_for_shot if c in curated_cols]
curated = pd.read_csv(CURATED_FULL_PATH, usecols=usecols)

direct_pii_present_in_curated = [c for c in direct_pii_cols if c in curated.columns]
pii_non_null_counts = {c: int(curated[c].notna().sum()) for c in direct_pii_present_in_curated}


# Redaction helpers 
def _is_na(v) -> bool:
    return v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v)


def _mask_text(v, replacement="[REDACTED]"):
    if _is_na(v):
        return v
    s = str(v).strip()
    return replacement if s != "" else v


def _mask_email(v):
    if _is_na(v):
        return v
    s = str(v).strip()
    if s == "":
        return v
    if "@" not in s:
        return "[REDACTED_EMAIL]"
    local, domain = s.split("@", 1)
    local_masked = (local[:1] + "***") if local else "***"
    return f"{local_masked}@{domain}"


def _mask_ssn(v):
    if _is_na(v):
        return v
    s = str(v).strip()
    return f"***-**-{s[-4:]}" if s else v


def _mask_ip(v):
    if _is_na(v):
        return v
    s = str(v).strip()
    return "[REDACTED_IP]" if s else v


def _mask_zip(v):
    if _is_na(v):
        return v
    s = str(v).strip()
    return "[REDACTED_ZIP]" if s else v


def _mask_dob(v):
    if _is_na(v):
        return v
    s = str(v).strip()
    if s == "":
        return v
    m = re.search(r"(19|20)\d{2}", s)
    year = m.group(0) if m else "XXXX"
    return f"{year}-**-**"


# Redacted curated preview (first 10)
preview = curated.head(10).copy()
for col in preview.columns:
    col_l = col.lower()
    if "ssn" in col_l:
        preview[col] = preview[col].apply(_mask_ssn)
    elif "email" in col_l:
        preview[col] = preview[col].apply(_mask_email)
    elif "ip_address" in col_l:
        preview[col] = preview[col].apply(_mask_ip)
    elif "date_of_birth" in col_l:
        preview[col] = preview[col].apply(_mask_dob)
    elif "zip_code" in col_l:
        preview[col] = preview[col].apply(_mask_zip)
    elif "full_name" in col_l or col_l.endswith("name"):
        preview[col] = preview[col].apply(lambda x: _mask_text(x, "[REDACTED_NAME]"))


# Monitoring metrics
metrics = {
    "analysis_rows": int(len(analysis)),
    "analysis_cols": int(analysis.shape[1]),
}

if "applicant_pseudo_id" in analysis.columns:
    s = analysis["applicant_pseudo_id"]
    metrics["missing_applicant_pseudo_id"] = int(s.isna().sum())
    metrics["unique_applicant_pseudo_id"] = int(s.nunique(dropna=True))
    metrics["duplicate_applicant_pseudo_id_count"] = int(s.duplicated().sum())
else:
    metrics["missing_applicant_pseudo_id"] = None
    metrics["unique_applicant_pseudo_id"] = None
    metrics["duplicate_applicant_pseudo_id_count"] = None

if "pseudo_id_source" in analysis.columns:
    pseudo_source_dist = (
        analysis["pseudo_id_source"]
        .fillna("MISSING")
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
        .assign(pct=lambda d: (100 * d["count"] / len(analysis)).round(2) if len(analysis) else 0.0)
        .reset_index(names="pseudo_id_source")
    )
else:
    pseudo_source_dist = pd.DataFrame(columns=["pseudo_id_source", "count", "pct"])

if "pseudo_id_fallback_used_flag" in analysis.columns:
    fb = pd.Series(analysis["pseudo_id_fallback_used_flag"]).fillna(False)
    fb_bool = fb if fb.dtype == bool else fb.astype(bool)
    fallback_used_count = int(fb_bool.sum())
    metrics["fallback_used_count"] = fallback_used_count
    metrics["fallback_used_rate_pct"] = round(100 * fallback_used_count / len(analysis), 2) if len(analysis) else 0.0
else:
    metrics["fallback_used_count"] = None
    metrics["fallback_used_rate_pct"] = None

if "is_canonical_for_analysis" in curated.columns:
    canon = curated["is_canonical_for_analysis"]
    if canon.dtype == object:
        canon = (
            canon.astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False})
        )
    metrics["curated_canonical_rate_pct"] = round(100 * float(pd.Series(canon).dropna().mean()), 2) if canon.dropna().shape[0] else None
else:
    metrics["curated_canonical_rate_pct"] = None


# Evidence summary 
summary = []
summary.append("=== Evidence Summary (A + B) ===")
summary.append(f"ANALYSIS  rows={len(analysis):,} cols={analysis.shape[1]:,}")
summary.append(f"CURATED   rows={len(curated):,} cols={curated.shape[1]:,}")
summary.append("")
summary.append("A) applications_analysis.csv checks")
summary.append(f"Required fields present: {required_present}")
summary.append(f"Direct PII columns present in ANALYSIS (should be []): {direct_pii_present_in_analysis}")
summary.append("")
summary.append("B) applications_curated_full.csv checks")
summary.append(f"Direct PII columns present in CURATED_FULL (should be non-empty): {direct_pii_present_in_curated}")
summary.append(f"Non-null counts for PII cols (CURATED_FULL): {pii_non_null_counts}")
summary.append("")
summary.append("Monitoring metrics")
summary.append(f"Fallback used: {metrics.get('fallback_used_count')} rows ({metrics.get('fallback_used_rate_pct')}%)")
summary.append(f"Missing applicant_pseudo_id: {metrics.get('missing_applicant_pseudo_id')}")
summary.append(f"Duplicate applicant_pseudo_id (note: may be expected across multiple applications): {metrics.get('duplicate_applicant_pseudo_id_count')}")
if metrics.get("curated_canonical_rate_pct") is not None:
    summary.append(f"Curated canonical rate (is_canonical_for_analysis): {metrics.get('curated_canonical_rate_pct')}%")
summary.append(f"Curated preview columns used (redacted): {list(preview.columns)}")

summary_text = "\n".join(summary)

print(summary_text)
print("\n=== Curated preview (redacted, 10 rows) ===")
display(preview)
print("\n=== Pseudo ID source distribution (analysis) ===")
display(pseudo_source_dist)

# Save outputs 
(OUT_DIR / "summary.txt").write_text(summary_text, encoding="utf-8")
(OUT_DIR / "analysis_columns.txt").write_text("\n".join(list(analysis.columns)), encoding="utf-8")
preview.to_csv(OUT_DIR / "curated_full_preview_redacted_10rows.csv", index=False)
pseudo_source_dist.to_csv(OUT_DIR / "pseudo_id_source_distribution.csv", index=False)
pd.DataFrame([metrics]).to_csv(OUT_DIR / "pseudonymization_metrics.csv", index=False)

=== Evidence Summary (A + B) ===
ANALYSIS  rows=500 cols=16
CURATED   rows=502 cols=14

A) applications_analysis.csv checks
Required fields present: {'applicant_pseudo_id': True, 'pseudo_id_source': True, 'pseudo_id_fallback_used_flag': True}
Direct PII columns present in ANALYSIS (should be []): []

B) applications_curated_full.csv checks
Direct PII columns present in CURATED_FULL (should be non-empty): ['raw_applicant_full_name', 'raw_applicant_email', 'raw_applicant_ssn', 'raw_applicant_ip_address', 'raw_applicant_date_of_birth', 'clean_email', 'clean_date_of_birth']
Non-null counts for PII cols (CURATED_FULL): {'raw_applicant_full_name': 502, 'raw_applicant_email': 495, 'raw_applicant_ssn': 497, 'raw_applicant_ip_address': 497, 'raw_applicant_date_of_birth': 497, 'clean_email': 495, 'clean_date_of_birth': 497}

Monitoring metrics
Fallback used: 5 rows (1.0%)
Missing applicant_pseudo_id: 0
Duplicate applicant_pseudo_id (note: may be expected across multiple applications): 2
Curated 

,application_row_id,application_id,raw_applicant_full_name,raw_applicant_email,raw_applicant_ssn,raw_applicant_ip_address,raw_applicant_date_of_birth,raw_applicant_zip_code,clean_email,clean_gender,clean_date_of_birth,clean_zip_code,clean_annual_income,is_canonical_for_analysis
0,0,app_200,[REDACTED_NAME],j***@hotmail.com,***-**-4340,[REDACTED_IP],2001-**-**,[REDACTED_ZIP],j***@hotmail.com,Male,2001-**-**,[REDACTED_ZIP],73000.0,True
1,1,app_037,[REDACTED_NAME],b***@yahoo.com,***-**-4784,[REDACTED_IP],1992-**-**,[REDACTED_ZIP],b***@yahoo.com,Male,1992-**-**,[REDACTED_ZIP],78000.0,True
2,2,app_215,[REDACTED_NAME],s***@mail.com,***-**-5178,[REDACTED_IP],1989-**-**,[REDACTED_ZIP],s***@mail.com,Male,1989-**-**,[REDACTED_ZIP],61000.0,True
3,3,app_024,[REDACTED_NAME],t***@protonmail.com,***-**-1833,[REDACTED_IP],1983-**-**,[REDACTED_ZIP],t***@protonmail.com,Male,1983-**-**,[REDACTED_ZIP],103000.0,True
4,4,app_184,[REDACTED_NAME],b***@aol.com,***-**-2475,[REDACTED_IP],1999-**-**,[REDACTED_ZIP],b***@aol.com,Male,1999-**-**,[REDACTED_ZIP],57000.0,True
5,5,app_275,[REDACTED_NAME],m***@outlook.com,***-**-4912,[REDACTED_IP],1982-**-**,[REDACTED_ZIP],m***@outlook.com,Female,1982-**-**,[REDACTED_ZIP],110000.0,True
6,6,app_099,[REDACTED_NAME],n***@outlook.com,***-**-2503,[REDACTED_IP],1990-**-**,[REDACTED_ZIP],n***@outlook.com,Male,1990-**-**,[REDACTED_ZIP],55000.0,True
7,7,app_246,[REDACTED_NAME],s***@gmail.com,***-**-1864,[REDACTED_IP],1991-**-**,[REDACTED_ZIP],s***@gmail.com,Female,1991-**-**,[REDACTED_ZIP],82000.0,True
8,8,app_042,[REDACTED_NAME],j***@gmail.com,***-**-5530,[REDACTED_IP],1990-**-**,[REDACTED_ZIP],j***@gmail.com,Male,1990-**-**,[REDACTED_ZIP],69000.0,False
9,9,app_348,[REDACTED_NAME],m***@hotmail.com,***-**-8400,[REDACTED_IP],1989-**-**,[REDACTED_ZIP],m***@hotmail.com,Male,1989-**-**,[REDACTED_ZIP],55000.0,True



=== Pseudo ID source distribution (analysis) ===


,pseudo_id_source,count,pct
0,ssn,495,99.0
1,name_dob_zip_fallback,4,0.8
2,email_fallback,1,0.2


# 3. Data Quality

In [5]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root (no 'data/' folder found up the tree).")


ROOT = find_repo_root(Path.cwd())

PRE_PATH = ROOT / "data" / "quality" / "data_quality_report.csv"
POST_PATH = ROOT / "data" / "quality" / "reports" / "post" / "data_quality_report_postclean.csv"
BA_PATH = ROOT / "data" / "quality" / "before_after_comparison.csv"

OUT_DIR = ROOT / "reports" / "Governance" / "data_quality_study"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [PRE_PATH, POST_PATH, BA_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")


# Load inputs
pre = pd.read_csv(PRE_PATH)
post = pd.read_csv(POST_PATH)
before_after = pd.read_csv(BA_PATH)


# Standardize column names
pre.columns = [c.strip() for c in pre.columns]
post.columns = [c.strip() for c in post.columns]

# POST sometimes uses issue_type instead of issue_group
if "issue_type" in post.columns and "issue_group" not in post.columns:
    post = post.rename(columns={"issue_type": "issue_group"})

# Governance helper outputs sometimes use affected_count / affected_pct
if "affected_count" in post.columns and "count" not in post.columns:
    post = post.rename(columns={"affected_count": "count"})
if "affected_pct" in post.columns and "percent" not in post.columns:
    post = post.rename(columns={"affected_pct": "percent"})


# Validate + clean PRE
required_pre = {"stage", "issue_group", "rule_id", "field_path", "description", "count", "percent", "severity", "value_source"}
missing_pre = required_pre - set(pre.columns)
if missing_pre:
    raise ValueError(f"PRE report missing columns: {sorted(missing_pre)}")

pre = pre[pre["stage"].astype(str).str.strip().str.lower().eq("pre")].copy()
pre["count"] = pd.to_numeric(pre["count"], errors="coerce")
pre["percent"] = pd.to_numeric(pre["percent"], errors="coerce")

pre_keep = ["rule_id", "issue_group", "field_path", "description", "severity", "value_source", "count", "percent"]
pre = pre[pre_keep].rename(columns={"count": "pre_count", "percent": "pre_percent"})


# Validate + clean POST
required_post = {"rule_id", "issue_group", "field_path", "description", "count", "percent", "severity"}
missing_post = required_post - set(post.columns)
if missing_post:
    raise ValueError(f"POST report missing columns: {sorted(missing_post)}")

post["count"] = pd.to_numeric(post["count"], errors="coerce")
post["percent"] = pd.to_numeric(post["percent"], errors="coerce")

post_keep = ["rule_id", "issue_group", "field_path", "description", "severity", "count", "percent"]
if "value_source" in post.columns:
    post_keep.append("value_source")

post = post[post_keep].rename(columns={"count": "post_count", "percent": "post_percent"})


# Keep value_source from both sides if present
if "value_source" in pre.columns and "value_source" in post.columns:
    pre = pre.rename(columns={"value_source": "pre_value_source"})
    post = post.rename(columns={"value_source": "post_value_source"})
elif "value_source" in pre.columns:
    pre = pre.rename(columns={"value_source": "pre_value_source"})
elif "value_source" in post.columns:
    post = post.rename(columns={"value_source": "post_value_source"})


# Merge + prefer POST metadata when PRE missing
cmp = pre.merge(post, on="rule_id", how="outer", suffixes=("_pre", "_post"))

for col in ["issue_group", "field_path", "description", "severity"]:
    pre_col, post_col = f"{col}_pre", f"{col}_post"
    if pre_col in cmp.columns and post_col in cmp.columns:
        cmp[col] = cmp[post_col].combine_first(cmp[pre_col])
        cmp = cmp.drop(columns=[pre_col, post_col])

for c in ["pre_count", "post_count", "pre_percent", "post_percent"]:
    if c in cmp.columns:
        cmp[c] = pd.to_numeric(cmp[c], errors="coerce")

cmp["delta_count"] = cmp["post_count"] - cmp["pre_count"]
cmp["delta_pp"] = cmp["post_percent"] - cmp["pre_percent"]


# Save main output + keep BA as evidence copy
cmp.to_csv(OUT_DIR / "pre_post_comparison.csv", index=False)
before_after.to_csv(OUT_DIR / "before_after_comparison.csv", index=False)


# Quick sanity checks (same set as no teu notebook)
check_rules = ["R_APP_006", "R_APP_008", "R_APP_012", "R_APP_013", "R_APP_014", "R_APP_001", "R_APP_009"]
print("\nSanity check (selected rules):")
print(
    cmp[cmp["rule_id"].isin(check_rules)][
        ["rule_id", "severity", "pre_count", "pre_percent", "post_count", "post_percent", "delta_pp"]
    ].sort_values("rule_id")
)


print("\nTop improvements (delta_pp ascending):")
top_improvements = cmp.sort_values("delta_pp").head(15)
print(top_improvements[["rule_id", "severity", "pre_percent", "post_percent", "delta_pp", "pre_count", "post_count"]])
top_improvements.to_csv(OUT_DIR / "top_improvements_15.csv", index=False)


print("\nResidual issues (post_percent > 0):")
residual = cmp[cmp["post_percent"].fillna(0) > 0].copy()
severity_order = {"high": 0, "medium": 1, "low": 2}
residual["severity_rank"] = residual["severity"].astype(str).str.strip().str.lower().map(severity_order).fillna(9)
residual = residual.sort_values(["severity_rank", "post_percent"], ascending=[True, False])

print(residual[["rule_id", "severity", "post_percent", "post_count", "description"]].head(20))
residual.head(200).to_csv(OUT_DIR / "residual_issues_top200.csv", index=False)


# Export residual table (selected rules) as evidence
residual_tbl = cmp[cmp["rule_id"].isin([
    "R_APP_001", "R_APP_009", "R_APP_002", "R_DUP_003", "R_APP_003", "R_DUP_001", "R_APP_004", "R_APP_005"
])][["rule_id", "severity", "post_percent", "post_count", "description"]].copy()

residual_tbl.to_csv(OUT_DIR / "residual_issues_selected.csv", index=False)


Sanity check (selected rules):
      rule_id severity  pre_count  pre_percent  post_count  post_percent  \
0   R_APP_001     high      440.0        87.65         440         87.65   
5   R_APP_006      low      111.0        22.11           0          0.00   
7   R_APP_008      low      157.0        31.27           0          0.00   
8   R_APP_009   medium       39.0         7.77          39          7.77   
11  R_APP_012     high        2.0         0.40           0          0.00   
12  R_APP_013     high        1.0         0.20           0          0.00   
13  R_APP_014     high        1.0         0.20           0          0.00   

    delta_pp  
0       0.00  
5     -22.11  
7     -31.27  
8       0.00  
11     -0.40  
12     -0.20  
13     -0.20  

Top improvements (delta_pp ascending):
      rule_id severity  pre_percent  post_percent  delta_pp  pre_count  \
7   R_APP_008      low        31.27          0.00    -31.27      157.0   
5   R_APP_006      low        22.11          0.00  